# 질문별 대화 맥락 압축 — 연구 노트 01
2026-10-22 · 운영 변경 없음 · 합성 실험
본문 표는 experiment.py의 저장된 CLI 실행 결과다. 아래 재현 코드는 이 노트북에서 아직 실행하지 않았다.


## 이번 연구의 결론과 범위


<p><strong>현재 운영 방식은 유지한다.</strong> 이 문서는 질문에 필요한 과거 맥락만 고르는 방법의 수학적 조건을 조사한 첫 연구 기록이다.
운영 코드·설정·데이터베이스·데몬을 변경하지 않았으며, 아래 실험은 실제 대화나 언어 모델을 사용하지 않았다.</p>
<p>확인한 핵심은 두 가지다. 반복된 정보에는 추가 가치를 줄이는 모델이 적합하지만, 함께 있어야 이해되는 정보에는 다른 구조가 필요하다.
그리고 정해진 점수를 가장 잘 조합하는 문제를 정확히 풀어도, 그 점수가 의미를 잘못 평가했다면 좋은 답변을 보장하지 못한다.</p>
<p>따라서 다음 연구 가설은 <strong>질문별 선택 + 명시적인 참조 관계 + 생략 판단의 불확실성</strong>이다.
아직 실제 시스템에 적용할 근거는 없다. 새로운 알고리즘의 독창성이나 실제 품질 향상을 주장하는 문서도 아니다.</p>


## 1. 어떤 최적화 문제인가


<p>현재 질문을 q, 반드시 유지하는 맥락을 P, 선택 가능한 표현들의 집합을 S라고 한다.
P에는 현재 메시지, 필요한 직전 문답, 최신 정정, 끝나지 않은 요청, 핵심 권한, 최신 실행 근거를 둔다.
P는 이 연구에서 이미 옳게 구성됐다고 가정한다. 실제로 이를 식별하는 일은 별도 미해결 문제다.</p>
<pre>b_f = P가 이미 제공하는 사실 f의 정도
w_f(q) ≥ 0 = 질문 q에서 사실 f가 갖는 중요도
a_if ∈ [0,1] = 표현 i가 사실 f를 제공하는 정도

G_q(S | P) = Σ_f w_f(q) [ max(b_f, max_{i∈S} a_if) − b_f ]
J_q(S) = G_q(S | P) − λ C(S)
S* = argmax_{S∈𝒜} J_q(S)</pre>
<p>빈 집합의 내부 최댓값은 0으로 둔다. C는 추가 프롬프트와 검색·평가 비용을 공통 효용 척도로 환산한 값이다.
실험에서는 임의의 비용 단위만 사용한다. 실제 토큰, 금액, 지연 시간은 따로 측정해야 한다.
𝒜는 원문별 하나의 표현만 선택하고, 필요한 참조 관계와 모델의 물리적 입력 한도를 만족하는 후보들이다.</p>
<p>과거 교환 7개에 생략·색인·요약·원문 4개 선택지를 허용하면 4⁷ = 16,384개 조합이다.
이 유한 후보 집합에서는 모두 비교할 수 있다. 전체 장기 기억에서의 전역 최적해나 의미의 정확성을 의미하지 않는다.</p>
<p>이 식은 의미를 평가하기 위한 <strong>대리 목적함수</strong>다. 실제 답변 정확도나 상호정보량과 같다고 가정하지 않는다.
문서 요약에 submodularity를 활용한 선행 연구는 이 모델의 출발점이다.
<a href='https://aclanthology.org/P11-1052/'>Lin &amp; Bilmes, ACL 2011</a>.</p>


## 2. 보장할 수 있는 성질과 보장할 수 없는 성질


<p>m_f(S) = max(b_f, max a_if)라고 쓰면 새 표현 v의 추가 효용은 다음과 같다.</p>
<pre>Δ_v G(S) = Σ_f w_f max(0, a_vf − m_f(S))</pre>
<p>A ⊆ B이면 m_f(A) ≤ m_f(B)다. 따라서 Δ_v G(A) ≥ Δ_v G(B).
고정된 비음수 가중치와 고정된 coverage 값 아래에서 G는 단조 증가하는 submodular 함수다.
같은 사실을 이미 충분히 전달했다면 추가 표현의 이득은 0이고, 현재 질문에서 가중치가 0인 사실도 이득이 없다.</p>
<p>그러나 비용을 뺀 J는 단조 증가하지 않을 수 있다. 비용이 단순 합계이면 submodularity는 유지되지만,
공통 포맷 비용처럼 묶음에 따라 달라지면 J의 submodularity도 일반적으로 유지되지 않는다.
표현별 배타 선택이나 참조 의존성까지 추가한 이 문제에 표준 탐욕법의 1−1/e 보장을 그대로 붙일 수 없다.</p>
<p>또한 a와 w가 부정확하면 위 정리는 <em>부정확한 점수</em>의 구조만 보장한다.
‘실행할 계획’과 ‘실행 완료’, ‘15시’와 ‘16시로 정정’은 주제가 비슷해도 다른 사실이다.
사실의 키에는 주체·속성·값·시점·진행 상태·출처의 구별이 필요하다.
현재 유효한 상태를 고르는 질문과 과거 변경 과정을 묻는 질문은 서로 다른 정보를 요구한다.</p>


## 3. 두 반례: 연결 관계와 공통 비용


<p><strong>정보를 함께 읽어야 가치가 생기는 경우.</strong> A와 B가 서로 독립인 공정한 비트이고 Y = A XOR B라면:</p>
<pre>I(Y; A) = 0 bit
I(Y; B) = 0 bit
I(Y; A,B) = 1 bit</pre>
<p>각각의 정보만 보면 가치가 없지만 둘을 알면 답이 결정된다. 이 반례는 Krause &amp; Guestrin의 Example 1을 재현했다.
그 논문에서 정보 이득의 submodularity 보장에는 조건부 독립성과 같은 조건이 필요하다.
<a href='https://www.cs.cmu.edu/~guestrin/Publications/UAI2005/uai2005.pdf'>Krause &amp; Guestrin, UAI 2005, §3.2</a>.</p>
<p>대화의 ‘아까 말한 세 번째 것’과 해당 목록도 비슷한 상보 관계를 가질 수 있다. 이 언어적 비유 자체를 실험으로 검증한 것은 아니다.
참조 문장과 그 대상을 묶어서 선택하는 방법을 연구할 수 있지만, 관계를 올바르게 알아내는 비용과 오차는 남는다.</p>
<p><strong>요약들이 공통 틀을 나눠 쓰는 경우.</strong> 항목마다 효용 0.6, 본문 비용 0.1,
하나라도 넣으면 공통 포맷 비용 0.9를 내고 λ = 1이라 하자.</p>
<div class='table-wrap'><table><thead><tr><th>선택 개수</th><th>효용</th><th>비용</th><th>순가치</th></tr></thead><tbody><tr><td>0</td><td>0</td><td>0</td><td>0</td></tr><tr><td>1</td><td>0.6</td><td>1.0</td><td>-0.4</td></tr><tr><td>2</td><td>1.2</td><td>1.1</td><td>0.1</td></tr></tbody></table></div>
<p>양의 한계 이득이 있는 항목만 하나씩 추가하는 탐욕법은 아무것도 고르지 않는다. 전체 비교는 둘을 고른다.
이는 공통 비용이 있는 경우의 자체 구성 반례이며, 현재 SOOM의 포맷 비용 수치를 측정한 결과는 아니다.</p>


## 4. 적응형 예산에서 증명되는 것


<p>λ를 고정하면 질문별 G가 달라지므로 선택 길이도 달라진다. 반대로 후보·효용·비용을 모두 고정한 상태에서
λ만 높이면 정확한 최적해가 선택하는 비용은 증가하지 않는다.</p>
<pre>λ₂ &gt; λ₁, S₁ ∈ argmax(G − λ₁C), S₂ ∈ argmax(G − λ₂C)
G(S₁) − λ₁C(S₁) ≥ G(S₂) − λ₁C(S₂)
G(S₂) − λ₂C(S₂) ≥ G(S₁) − λ₂C(S₁)
두 식을 더하면 (λ₂ − λ₁)(C(S₁) − C(S₂)) ≥ 0
따라서 C(S₂) ≤ C(S₁).</pre>
<p>아래는 같은 가상 API·결제 질문에서 λ만 바꾼 결과다.</p>
<div class='table-wrap'><table><thead><tr><th>비용 가중치 λ</th><th>선택 비용</th><th>보존한 추가 효용</th></tr></thead><tbody><tr><td>0.0</td><td>570</td><td>9.0</td></tr><tr><td>0.0005</td><td>570</td><td>9.0</td></tr><tr><td>0.001</td><td>570</td><td>9.0</td></tr><tr><td>0.003</td><td>210</td><td>8.1</td></tr><tr><td>0.01</td><td>210</td><td>8.1</td></tr><tr><td>0.02</td><td>210</td><td>8.1</td></tr><tr><td>0.05</td><td>0</td><td>0.0</td></tr><tr><td>0.1</td><td>0</td><td>0.0</td></tr></tbody></table></div>
<p>선택 길이가 계단식으로 줄어드는 성질은 확인했지만, 어느 λ가 실제 사용자에게 최선인지는 이 정리로 알 수 없다.
질문마다 점수를 다시 매기거나 후보가 바뀌는 실제 운영의 길이까지 단조 감소한다고 주장할 수도 없다.</p>


## 5. 의미 평가 오차가 최적화보다 앞선다


<p>참된 목적함수를 J, 추정한 함수를 Ĵ라 하자. 같은 유한 후보 전체에서 |Ĵ(S) − J(S)| ≤ ε가 보장되고,
S*가 참값 최적해, Ŝ가 추정값 최적해라면:</p>
<pre>J(S*) − J(Ŝ)
 = [J(S*) − Ĵ(S*)] + [Ĵ(S*) − Ĵ(Ŝ)] + [Ĵ(Ŝ) − J(Ŝ)]
 ≤ ε + 0 + ε = 2ε.</pre>
<p>즉 최적화는 정확해도 최종 목적값 손실은 의미 점수 오차에 달려 있다. 보호 맥락 b와 비용이 알려져 있고
a, â ∈ [0,1], w, ŵ ≥ 0이면 다음은 ε의 충분한 상계다.</p>
<pre>ε ≤ ‖ŵ − w‖₁ + Σ_f w_f · max_i |â_if − a_if|</pre>
<p>최댓값 연산은 각 좌표의 최대 오차보다 크게 변하지 않고, coverage의 증가량이 [0,1] 안에 있으므로 성립한다.
비용도 추정하면 λ·sup_S|Ĉ(S)−C(S)| 항이 추가된다. 보호 맥락이나 후보 집합 자체를 잘못 구성한 오류는 이 식이 해결하지 않는다.</p>
<p><strong>다음 가설:</strong> 기존 표현 S₀를 더 짧은 S로 바꿀 때
Ĵ(S)−Ĵ(S₀) &gt; ε(S)+ε(S₀)라는 근거가 있으면, 그 오차 범위 안에서는 목적값이 개선된다.
그런 근거를 얻지 못한 경우 원문 유지나 필요 시 검색을 허용하는 보수적 선택을 검토한다.
이것은 조건부 명제이며, 언어 모델이 말하는 자신감이 유효한 오차 상계라는 뜻은 아니다.</p>


## 6. 재현한 합성 실험


<p><strong>손으로 정한 네 가지 사례.</strong> 선택지 7개 × 표현 4종의 모든 조합을 비교했다.
색인은 이 한 단계 실험에서 답변 coverage를 주지 않으므로 생략에 지배된다.
색인이 다음 검색을 돕는 가치는 아직 모델링하지 않았다.</p>
<div class='table-wrap'><table><thead><tr><th>가상 질문</th><th>선택한 과거 정보</th><th>추가 비용 단위</th><th>효용 − 비용</th></tr></thead><tbody><tr><td>현재 프롬프트 상태만 질문</td><td>없음</td><td>0</td><td>0.000</td></tr><tr><td>API 후속 질문</td><td>api:summary</td><td>120</td><td>4.140</td></tr><tr><td>API와 결제에 함께 질문</td><td>api:summary, payments:summary</td><td>210</td><td>7.470</td></tr><tr><td>특정 절의 원문 그대로 요청</td><td>section_quote:original</td><td>2100</td><td>3.700</td></tr></tbody></table></div>
<p>현재 상태가 보호 맥락에 충분히 있을 때 과거 정보 0개, 관련 후속 질문에는 관련 요약,
정확한 인용에는 원문을 고르게 할 수 있다. 이는 수식이 정한 점수대로 작동한다는 확인이며,
자연어 질문에서 이 점수를 정확히 추론했다는 결과가 아니다.</p>
<p><strong>점수 오차 민감도.</strong> 고정 시드 20261022로 80개 가상 세계를 만들었다.
각 세계는 사실 6개, 과거 항목 5개, 표현 4종과 알려진 보호 맥락을 가진다.
정답 coverage와 중요도를 먼저 생성한 다음, 중요도와 요약·원문 coverage 각각에 N(0,σ²) 잡음을 더해 [0,1]로 잘랐다.
같은 세계는 각 σ에서 같은 기초 난수를 사용한다. 이는 실제 임베딩이나 LLM의 오류 분포를 추정한 것이 아니다.</p>
<p>세계마다 1,024개 조합을 정확히 비교했다. 아래 손실은 참된 목적값의 최댓값에서 추정 점수로 선택한 조합의 참된 목적값을 뺀 값이다.</p>
<div class='table-wrap'><table><thead><tr><th>점수 잡음 σ</th><th>평균 목적값 손실</th><th>최대 목적값 손실</th><th>손실 발생 / 80개 세계</th></tr></thead><tbody><tr><td>0.00</td><td>0.0000</td><td>0.0000</td><td>0</td></tr><tr><td>0.05</td><td>0.0099</td><td>0.1151</td><td>20</td></tr><tr><td>0.10</td><td>0.0375</td><td>0.3522</td><td>33</td></tr><tr><td>0.20</td><td>0.0993</td><td>0.6078</td><td>50</td></tr><tr><td>0.40</td><td>0.2551</td><td>1.5324</td><td>65</td></tr></tbody></table></div>
<p>80개 세계를 5개 잡음 수준에서 평가한 총 400개 조건에서 regret ≤ 2ε와 위 ε 상계를 확인했다.
이는 400개의 독립 실제 대화 평가가 아니다. σ=0에서 손실 0은 정확한 탐색 구현의 기본 확인이다.
이 시드에서는 잡음이 커지며 평균 손실이 커졌지만, 이를 일반적인 단조성 정리나 실제 답변 오류율로 해석하면 안 된다.</p>
<p>합성 정답을 알고 있기 때문에 ε를 계산할 수 있었다. 실제 운용에서는 알 수 없는 값이다.
특히 σ=0.4의 한 세계에서는 추정 선택의 참된 순가치가 음수가 되어, 아무것도 추가하지 않는 선택보다 나빴다.
어떤 정보가 중요하고 어떤 요약이 충분한지를 검증하는 평가가 필요한 이유다.</p>


## 7. 보편적인 무손실 요약은 가능한가


<p>검색할 원문 없이, 가능한 모든 미래 질문에 정확히 답하는 더 짧은 요약은 일반적으로 불가능하다.
n비트로 구별되는 임의의 기록 2ⁿ개를 고정 길이 b비트(b&lt;n) 표현 2ᵇ개로 옮기면 같은 표현을 공유하는 서로 다른 기록이 있다.
그 두 기록에서 다른 비트의 값을 묻는 질문에는 같은 요약만으로 항상 맞힐 수 없다.</p>
<p>이는 단순한 비둘기집 원리의 결과다. 실제 자연어가 항상 압축 불가능하다는 뜻은 아니다.
보장 대상을 <strong>현재 질문 또는 정해진 질문 분포</strong>로 좁히고, 필요한 경우 원문에 접근하도록 하면
짧은 기본 맥락과 상세 정보 보존을 함께 연구할 수 있다. 이때 검색 실패와 추가 지연도 전체 비용에 들어간다.</p>
<p>RECOMP는 검색된 내용이 입력에 도움이 되지 않을 때 빈 압축 결과를 허용하는 선행 연구다.
해당 논문의 언어 모델링·질의응답 성과를 SOOM의 대화 기억 성과로 옮겨 주장하지 않는다.
<a href='https://arxiv.org/abs/2310.04408'>Xu, Shi &amp; Choi, RECOMP</a>.</p>


## 8. 다음에 검증할 연구 질문


<p>연구 순서는 우선 ‘무엇을 빼도 되는지’를 평가하는 작은 기준 집합, 다음으로 선택 방식 비교다.
반복 설명, 주제 전환, 지시어와 목록, 최신 정정, 계획과 완료, 정확한 인용, 미완료 약속, 근거 부족을 포함한다.
합성 사례와 실제 대화 평가는 별도 결과로 관리하고, 같은 대화의 바꿔 쓴 문장이 학습·평가 양쪽에 섞이지 않도록 분리한다.</p>
<p>비교군은 현재 방식, 단순 최근 대화, 질문별 선택, 참조 묶음을 포함한 선택으로 둔다.
평가는 원문에 근거한 답과 출처를 기준으로 하고, 전체 프롬프트를 받은 모델의 답 자체를 정답으로 간주하지 않는다.
핵심 사실·정정·미완료 요청 보존, 정확한 인용, 잘못된 단정과 보류를 따로 측정한다.</p>
<p>품질과 함께 본 응답의 실제 입력 토큰, 평가·요약·검색의 모든 추가 호출, 첫 사용과 캐시 재사용 지연을 함께 기록한다.
현재까지 실제 언어 모델 비교, 벤치마크 실행, 비용 절감 검증은 하지 않았다.</p>
<p>대화 평가의 참고 자료로 LongMemEval은 정보 추출·여러 세션 추론·시간 추론·지식 갱신·보류를 구분한다.
<a href='https://arxiv.org/abs/2410.10813'>Wu et al., LongMemEval, ICLR 2025</a>.
더 최근의 <a href='https://xiaowu0162.github.io/longmemeval-v2/'>LongMemEval-V2 저자 페이지</a>는 웹 에이전트 실행 이력과 정확도·검색 지연을 다룬다.
SOOM의 일상 대화에는 전자의 평가 유형이 더 직접적인 출발점이라는 것이 이번 연구의 판단이다.
실제 실험을 시작할 때 데이터 버전과 평가 코드를 고정해야 한다.</p>
<p>추가 평가·검색 비용이 절감분을 소모하거나 중요한 정정과 약속의 누락을 줄이지 못하면 적용하지 않는다.
연구 결과가 유용하지 않아도 현재 운용에는 영향이 없도록 유지한다.</p>


## 재현 및 증거 파일


<p>이 결과는 Python 3.12.13 표준 라이브러리만으로 CLI에서 생성했다.
한 번의 실행 시간은 1.014초이며 속도 벤치마크가 아니다.
노트북의 재현 셀은 실행하지 않은 상태로 제공하며, 본문 표는 저장된 CLI 실험 결과다.</p>
<pre>python -X utf8 experiment.py
python -X utf8 make_note.py</pre>
<p><a href='experiment.py'>실험 코드</a> · <a href='results.json'>원시 결과</a> ·
<a href='research.ipynb'>연구 노트북</a> · <a href='make_note.py'>문서 생성 코드</a></p>
<p class='small'>experiment.py SHA-256: 2a8b73a148413d0595077b223b334caafcf9699d2d5af419e5ae1d8dee4cdb88<br>
문헌 확인일: 2026-10-22. 운영 파일이나 원문 대화는 이 연구 묶음에 복사하지 않았다.</p>


In [ ]:
# 이 노트북과 experiment.py가 있는 폴더를 작업 디렉터리로 사용한다.
from experiment import run_all, summary
import json
result = run_all()
print(json.dumps(summary(result), ensure_ascii=False, indent=2))
